# 01 — Business Understanding and Exploratory Data Analysis

**Capstone Project 24.1 | Enterprise SaaS Churn Risk**

## Business problem
Churn and downgrade can reduce recurring revenue and customer lifetime value. Customer Success teams need a repeatable way to identify accounts for proactive review. This project asks whether account profile, product adoption, engagement and value-realization indicators can distinguish churn outcomes.

> **Data transparency:** This repository uses a synthetic teaching dataset of 3,000 simulated SaaS accounts. It is not customer telemetry. The target is generated from an explicit simulation mechanism involving the same broad risk indicators studied in this notebook. Results are educational demonstrations, not real-world predictions or industry benchmarks.

In [ ]:
from pathlib import Path
import runpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data/raw/enterprise_saas_churn_synthetic.csv'
if not DATA.exists():
    runpy.run_path(str(ROOT / 'scripts/generate_synthetic_dataset.py'))
df = pd.read_csv(DATA)
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

## 1. Data quality and target balance

The account identifier is excluded from modeling. Missing values are retained for pipeline-based imputation. Duplicate account IDs are checked before analysis.

In [ ]:
quality = pd.DataFrame({'missing_count':df.isna().sum(),'missing_pct':(df.isna().mean()*100).round(2),'unique_values':df.nunique()})
display(quality.sort_values('missing_pct',ascending=False))
print('Duplicate account IDs:',df.account_id.duplicated().sum())
display(df.churned.value_counts().rename(index={0:'Retained',1:'Churned'}).to_frame('accounts'))
print(f'Simulated churn prevalence: {df.churned.mean():.1%}')

## 2. Categorical and continuous exploration

The following charts summarize class balance, churn rate by customer tier and the distribution of important numeric variables. Unusual enterprise ARR or usage values are not automatically removed because they may be legitimate.

In [ ]:
IMG = ROOT / 'images'; IMG.mkdir(exist_ok=True)
fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.countplot(x=df.churned.astype(str),color='steelblue',ax=axes[0])
axes[0].set(title='Simulated account outcome distribution',xlabel='Churn outcome (0 retained, 1 churned)',ylabel='Accounts')
tier_rate=df.groupby('customer_tier',observed=True).churned.mean().reset_index()
sns.barplot(data=tier_rate,x='customer_tier',y='churned',color='slateblue',ax=axes[1])
axes[1].set(title='Simulated churn rate by tier',xlabel='Customer tier',ylabel='Share churned',ylim=(0,1))
plt.tight_layout(); plt.savefig(IMG/'eda_target_and_tier.png',dpi=160); plt.show()
features=['tenure_months','annual_recurring_revenue_usd','license_utilization_pct','dau_mau_ratio','active_modules','support_cases_last_90d','days_to_renewal']
fig,axes=plt.subplots(3,3,figsize=(14,10)); axes=axes.flatten()
for ax,col in zip(axes,features):
    sns.histplot(data=df,x=col,hue='churned',stat='density',common_norm=False,element='step',ax=ax)
    ax.set_title(col.replace('_',' ').title())
for ax in axes[len(features):]: ax.remove()
plt.tight_layout(); plt.savefig(IMG/'eda_numeric_distributions.png',dpi=160); plt.show()

In [ ]:
corr_cols=['tenure_months','annual_recurring_revenue_usd','license_utilization_pct','dau_mau_ratio','active_modules','monthly_api_calls','qbrs_attended_last_4','csat_score','kpi_milestones_achieved_pct','automated_workflows','support_cases_last_90d','days_to_renewal','churned']
fig,ax=plt.subplots(figsize=(12,9))
sns.heatmap(df[corr_cols].corr(numeric_only=True),cmap='vlag',center=0,ax=ax)
ax.set_title('Numeric feature correlation matrix')
plt.tight_layout(); plt.savefig(IMG/'eda_correlation_heatmap.png',dpi=160); plt.show()
for col in ['customer_tier','industry','region']:
    display(df.groupby(col,observed=True).churned.agg(accounts='size',churn_rate='mean').sort_values('churn_rate',ascending=False))

## 3. Initial interpretation and limitations

The simulated label mechanism deliberately makes lower adoption, weaker engagement, lower satisfaction/value realization and higher support burden associated with greater churn probability. This lets the modeling workflow demonstrate signal recovery. It does **not** prove that these relationships or rates hold in a real SaaS portfolio. Notebook 03 evaluates how well classifiers recover the simulated outcome on held-out records.